# AI-Driven Market Analysis for Computer Component Price Surge

**Decision Support System for Hardware Procurement in the AI Era**

Pipeline: Scraping → Preprocessing → Statistical Analysis → Sentiment → AHP-TOPSIS → Visualization.

> Run cells top to bottom. CPU runtime is sufficient.

## 1. Environment Setup

In [1]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(
        ["git", "clone", "https://github.com/bugkey24/ai-era-pc-component-market-analysis.git"]
    )
    get_ipython().run_line_magic("cd", "ai-era-pc-component-market-analysis")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import nltk

nltk.download("stopwords", quiet=True)
print("Environment ready. IN_COLAB =", IN_COLAB)

Environment ready. IN_COLAB = False


## 2. Configuration & Imports

In [2]:
from src.utils import load_config, setup_logger

CONFIG_PATH = "config.yaml"
config = load_config(CONFIG_PATH)
logger = setup_logger(name="notebook", level=config["logging"]["level"])
config["scraping"]["platforms"]

[{'name': 'tokopedia',
  'enabled': True,
  'method': 'static',
  'base_url': 'https://www.tokopedia.com',
  'reviews_enabled': True,
  'search_keywords': {'gpu': 'rtx', 'ram': 'ddr5', 'ssd': 'nvme'}},
 {'name': 'shopee',
  'enabled': True,
  'method': 'dynamic',
  'base_url': 'https://shopee.co.id',
  'reviews_enabled': True},
 {'name': 'blibli',
  'enabled': True,
  'method': 'static',
  'base_url': 'https://www.blibli.com',
  'reviews_enabled': False}]

## 3. Option A — Load Existing Data

Skip live scraping (platform markup changes often). Place CSVs in `data/raw/` or use the sample generator below.

In [3]:
from pathlib import Path

import pandas as pd

# Real experiment data ships with the repo (data/snapshot/);
# fresher local runs in data/raw take precedence, synthetic sample is the last resort.
candidates = [
    sorted(Path("data/snapshot").glob("*products*.csv")),
    sorted(Path("data/raw").glob("*products*.csv")),
]
csvs = next((c for c in candidates if c), [])

if csvs:
    data = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    print(f"Loaded {len(data)} rows from {len(csvs)} file(s): {[f.name for f in csvs]}")
else:
    # Minimal sample so the pipeline is runnable end-to-end offline
    import numpy as np

    rng = np.random.default_rng(42)
    rows = []
    for cat, base, n in [("gpu", 8_000_000, 20), ("ram", 1_400_000, 20), ("ssd", 800_000, 20)]:
        for i in range(n):
            rows.append(
                {
                    "product_id": f"{cat.upper()}-{i:03d}",
                    "name": f"Sample {cat.upper()} Model {i} {rng.choice(['8GB', '16GB', '512GB', '1TB'])}",
                    "category": cat,
                    "price": f"Rp {base * rng.uniform(0.6, 1.6):,.0f}",
                    "rating": round(rng.uniform(3.8, 5.0), 1),
                    "review_count": int(rng.integers(5, 400)),
                    "seller_rating": round(rng.uniform(4.0, 5.0), 1),
                    "seller_followers": int(rng.integers(10, 5000)),
                    "source": str(rng.choice(["tokopedia", "shopee", "blibli"])),
                }
            )
    data = pd.DataFrame(rows)
    print(f"Generated {len(data)} sample rows (no snapshot or raw CSVs found)")
data.head()

Loaded 247 rows from 1 file(s): ['tokopedia_products.csv']


,product_id,name,url,price,original_price,discount,rating,review_count,seller_name,seller_tier,location,source,category
0,100702835200,ZOTAC GAMING NVIDIA GeForce RTX 5070 Ti SOLID ...,https://www.tokopedia.com/gamingpcstore/zotac-...,25288000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
1,100702919487,MSI NVIDIA GEFORCE RTX 5050 8GB VENTUS 2X OC G...,https://www.tokopedia.com/gamingpcstore/msi-nv...,8955000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
2,100552637943,MANLI STELLAR NVIDIA GeForce RTX 5080 OC 16GB ...,https://www.tokopedia.com/gamingpcstore/manli-...,31900000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
3,100276456043,ZOTAC GAMING NVIDIA GeForce RTX 5060 Ti 16GB T...,https://www.tokopedia.com/gamingpcstore/zotac-...,15486000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
4,100493453963,Gainward NVIDIA GeForce RTX 5060 Python III 8G...,https://www.tokopedia.com/gamingpcstore/gainwa...,10730000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu


## 4. Option B — Live Scraping

⚠️ **Expect failures from Colab.** Colab runs on Google datacenter IPs, which e-commerce platforms throttle or block: Tokopedia typically times out, Blibli returns 403, and Shopee additionally needs a Chrome install (`!apt-get -qq update && apt-get -qq install chromium chromium-driver`). Our validated collections ran from a **residential IP** — see `docs/09-live-experiment-results.md`. For results, use the snapshot in cell above; only run this cell to experiment, Tokopedia-first, with `max_pages` small. The scrapers retry timeouts (3×, exponential backoff) and every request passes the robots guard.

In [4]:
# Optional — see the warnings above. Tokopedia-first, small scale.
from src import PipelineOrchestrator  # self-contained: no earlier-cell dependency

pipeline = PipelineOrchestrator(CONFIG_PATH)
data = pipeline._run_scraping()
data.head()

Failed on page 1: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6385d17f5+5855]
	chromedriver!(No symbol) [0x7ff6384f83a0]
	chromedriver!(No symbol) [0x7ff638325d7d]
	chromedriver!(No symbol) [0x7ff638380d2e]
	chromedriver!(No symbol) [0x7ff63838103c]
	chromedriver!(No symbol) [0x7ff6383d1c67]
	chromedriver!(No symbol) [0x7ff6383ce828]
	chromedriver!(No symbol) [0x7ff63837316b]
	chromedriver!(No symbol) [0x7ff6383740a3]
	chromedriver!GetHandleVerifier [0x7ff6389f6c9b+42acfb]
	chromedriver!GetHandleVerifier [0x7ff638a22422+456482]
	chromedriver!GetHandleVerifier [0x7ff638a163ce+44a42e]
	chromedriver!GetHandleVerifier [0x7ff6386cd87e+1018de]
	chromedriver!(No symbol) [0x7ff638504e9c]
	chromedriver!(No symbol) [0x7ff638500ac4]
	chromedriver!(No symbol) [0x7ff638500c54]
	chromedriver!(No symbol) [0x7ff6384ec24c]
	KERNEL32!BaseThreadInitThunk [0x7ffde1dbccb7+17]
	ntdll!RtlUserThreadStart [0x7ffde27ecaec+2c]



Failed on page 1: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6385d17f5+5855]
	chromedriver!(No symbol) [0x7ff6384f83a0]
	chromedriver!(No symbol) [0x7ff638325d7d]
	chromedriver!(No symbol) [0x7ff638380d2e]
	chromedriver!(No symbol) [0x7ff63838103c]
	chromedriver!(No symbol) [0x7ff6383d1c67]
	chromedriver!(No symbol) [0x7ff6383ce828]
	chromedriver!(No symbol) [0x7ff63837316b]
	chromedriver!(No symbol) [0x7ff6383740a3]
	chromedriver!GetHandleVerifier [0x7ff6389f6c9b+42acfb]
	chromedriver!GetHandleVerifier [0x7ff638a22422+456482]
	chromedriver!GetHandleVerifier [0x7ff638a163ce+44a42e]
	chromedriver!GetHandleVerifier [0x7ff6386cd87e+1018de]
	chromedriver!(No symbol) [0x7ff638504e9c]
	chromedriver!(No symbol) [0x7ff638500ac4]
	chromedriver!(No symbol) [0x7ff638500c54]
	chromedriver!(No symbol) [0x7ff6384ec24c]
	KERNEL32!BaseThreadInitThunk [0x7ffde1dbccb7+17]
	ntdll!RtlUserThreadStart [0x7ffde27ecaec+2c]



Failed on page 1: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6385d17f5+5855]
	chromedriver!(No symbol) [0x7ff6384f83a0]
	chromedriver!(No symbol) [0x7ff638325d7d]
	chromedriver!(No symbol) [0x7ff638380d2e]
	chromedriver!(No symbol) [0x7ff63838103c]
	chromedriver!(No symbol) [0x7ff6383d1c67]
	chromedriver!(No symbol) [0x7ff6383ce828]
	chromedriver!(No symbol) [0x7ff63837316b]
	chromedriver!(No symbol) [0x7ff6383740a3]
	chromedriver!GetHandleVerifier [0x7ff6389f6c9b+42acfb]
	chromedriver!GetHandleVerifier [0x7ff638a22422+456482]
	chromedriver!GetHandleVerifier [0x7ff638a163ce+44a42e]
	chromedriver!GetHandleVerifier [0x7ff6386cd87e+1018de]
	chromedriver!(No symbol) [0x7ff638504e9c]
	chromedriver!(No symbol) [0x7ff638500ac4]
	chromedriver!(No symbol) [0x7ff638500c54]
	chromedriver!(No symbol) [0x7ff6384ec24c]
	KERNEL32!BaseThreadInitThunk [0x7ffde1dbccb7+17]
	ntdll!RtlUserThreadStart [0x7ffde27ecaec+2c]



Failed on page 1: 404 Client Error: Not Found for url: https://www.blibli.com/c/gpu


Failed on page 1: 404 Client Error: Not Found for url: https://www.blibli.com/c/ram


Failed on page 1: 404 Client Error: Not Found for url: https://www.blibli.com/c/ssd


2026-09-04 02:37:50,828 - pipeline - INFO - Scraping complete — 351 rows


,product_id,name,url,price,original_price,discount,rating,review_count,seller_name,seller_tier,location,source,category
0,100702919487,MSI NVIDIA GEFORCE RTX 5050 8GB VENTUS 2X OC G...,https://www.tokopedia.com/gamingpcstore/msi-nv...,8955000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
1,100702835200,ZOTAC GAMING NVIDIA GeForce RTX 5070 Ti SOLID ...,https://www.tokopedia.com/gamingpcstore/zotac-...,25288000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
2,100552637943,MANLI STELLAR NVIDIA GeForce RTX 5080 OC 16GB ...,https://www.tokopedia.com/gamingpcstore/manli-...,31900000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
3,100276456043,ZOTAC GAMING NVIDIA GeForce RTX 5060 Ti 16GB T...,https://www.tokopedia.com/gamingpcstore/zotac-...,15486000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu
4,100493453963,Gainward NVIDIA GeForce RTX 5060 Python III 8G...,https://www.tokopedia.com/gamingpcstore/gainwa...,10730000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu


## 5. Preprocessing & Feature Engineering

In [5]:
from src.preprocessing import DataPreprocessor, FeatureEngineer

clean = (
    DataPreprocessor(data)
    .clean_prices()
    .handle_missing(config["preprocessing"].get("handle_missing", "drop"))
    .extract_specifications()
    .remove_outliers(threshold=config["preprocessing"].get("outlier_threshold", 3.0))
)
fe = FeatureEngineer(clean.df)
clean = fe.create_price_per_gb().create_weighted_rating().create_seller_trust_score()
df = clean.get_engineered_data()
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

344 rows, 18 columns


,product_id,name,url,price,original_price,discount,rating,review_count,seller_name,seller_tier,location,source,category,spec_capacity,spec_memory_type,spec_interface,price_per_gb,weighted_rating
0,100702919487,MSI NVIDIA GEFORCE RTX 5050 8GB VENTUS 2X OC G...,https://www.tokopedia.com/gamingpcstore/msi-nv...,8955000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,8GB,,,1119375.0,0.0
1,100702835200,ZOTAC GAMING NVIDIA GeForce RTX 5070 Ti SOLID ...,https://www.tokopedia.com/gamingpcstore/zotac-...,25288000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,16GB,,,1580500.0,0.0
2,100552637943,MANLI STELLAR NVIDIA GeForce RTX 5080 OC 16GB ...,https://www.tokopedia.com/gamingpcstore/manli-...,31900000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,16GB,,,1993750.0,0.0
3,100276456043,ZOTAC GAMING NVIDIA GeForce RTX 5060 Ti 16GB T...,https://www.tokopedia.com/gamingpcstore/zotac-...,15486000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,16GB,,,967875.0,0.0
4,100493453963,Gainward NVIDIA GeForce RTX 5060 Python III 8G...,https://www.tokopedia.com/gamingpcstore/gainwa...,10730000,0,0,5.0,0,STAR17 COM,2,Jakarta Selatan,tokopedia,gpu,8GB,,,1341250.0,0.0


## 6. Statistical Analysis

In [6]:
from src.analysis import StatisticalAnalyzer

stats_an = StatisticalAnalyzer(df)
summary = stats_an.describe().get_summary()
display(summary.round(2))
stats_an.correlation_matrix()
display(stats_an.price_trend_by_category())
stats_an.normality_test("price")

,count,mean,std,min,25%,50%,75%,max,median,iqr,skewness,kurtosis
price,344.0,10667217.62,11439537.40,39000.00,3468750.00,7233000.00,14177250.00,74311000.00,7233000.00,10708500.00,2.51,7.89
original_price,344.0,3316168.90,9581811.24,0.00,0.00,0.00,0.00,69999000.00,0.00,0.00,3.85,17.16
discount,344.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
rating,344.0,4.99,0.08,3.50,5.00,5.00,5.00,5.00,5.00,0.00,-17.99,329.19
review_count,344.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
seller_tier,344.0,2.00,0.00,2.00,2.00,2.00,2.00,2.00,2.00,0.00,0.00,0.00
price_per_gb,316.0,609870.95,706919.15,2244.14,4172.27,295890.62,1116515.62,3449916.67,295890.62,1112343.36,1.10,0.93
weighted_rating,344.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


,mean,median,std,min,max,count,cv
category,,,,,,,
gpu,18616036.20,15486000.0,13701170.75,2750000,74311000,143,0.736
ram,8122110.51,8499000.0,4678610.77,1425000,20950000,63,0.576
ssd,3592295.09,3412750.0,1950955.17,39000,9192000,138,0.543


{'statistic': 0.7302, 'p_value': 0.0, 'normal_at_0.05': False}

## 7. Sentiment Analysis

The SVM trains automatically when review CSVs exist in data/raw/reviews_*.csv (produced by the review scrapers; the pipeline merges per-product scores). Labels are weak supervision from review ratings. If the corpus is skewed (e-commerce reality: mostly positive), the model fits on full data and accuracy is reported as not measurable. A demo set below keeps this notebook runnable standalone.

In [7]:
from src.analysis import SentimentAnalyzer

demo_texts = [
    "harga mahal sekali",
    "terlalu mahal untuk spesifikasi ini",
    "harga naik terus",
    "harga murah dan terjangkau",
    "murah bagus worth it",
    "harga oke murah",
    "performa cepat stabil",
    "cepat dan stabil untuk AI training",
    "performa mantap",
    "lambat dan sering hang",
    "performa jelek lambat",
    "lemot sering ngehang",
] * 3
demo_labels = (["negative"] * 3 + ["positive"] * 3 + ["positive"] * 3 + ["negative"] * 3) * 3

sentiment = SentimentAnalyzer(language=config["sentiment"]["language"])
sentiment.train(demo_texts, demo_labels)
print(f"Demo-model accuracy: {sentiment.accuracy:.2f}  (replace with real labelled reviews)")

Demo-model accuracy: 1.00  (replace with real labelled reviews)


## 8. AHP-TOPSIS Decision Model

In [8]:
import numpy as np

from src.dss import AHPProcessor, TOPSISProcessor

dss_cfg = config["dss"]

ahp = AHPProcessor(dss_cfg["criteria"])
ahp.build_pairwise_matrix(dss_cfg["pairwise_matrix"])
ahp.calculate_weights().check_consistency()
print(ahp.summary())
assert ahp.is_consistent(), "Pairwise matrix inconsistent (CR >= 0.1) — revise config"

{'criteria': ['price', 'performance', 'rating', 'seller_reliability', 'sentiment', 'future_value'], 'weights': {'price': np.float64(0.227), 'performance': np.float64(0.4387), 'rating': np.float64(0.0918), 'seller_reliability': np.float64(0.0413), 'sentiment': np.float64(0.0413), 'future_value': np.float64(0.1598)}, 'lambda_max': 6.3633, 'consistency_ratio': 0.0586, 'is_consistent': True}


In [9]:
# Decision matrix: map config criteria to available columns.
# Column availability differs by dataset (live cache has seller_tier but no
# seller_rating/followers; synthetic demo data is the reverse) — resolve
# with fallbacks instead of assuming.
def col_for(criterion: str) -> str:
    candidates = {
        "price": ["price"],
        "performance": ["rating"],
        "rating": ["weighted_rating", "rating"],
        "seller_reliability": ["seller_tier", "seller_trust", "rating"],
        "sentiment": ["sentiment_score", "rating"],
        "future_value": ["price_per_gb"],
    }[criterion]
    return next((c for c in candidates if c in df.columns), candidates[-1])


matrix = np.column_stack(
    [pd.to_numeric(df[col_for(c)], errors="coerce").fillna(0) for c in dss_cfg["criteria"]]
)

topsis = TOPSISProcessor(matrix, ahp.get_weights(), dss_cfg["criteria_types"])
ranking = topsis.rank()
df_ranked = (
    df.reset_index(drop=True)
    .loc[ranking["Alternative"]]
    .assign(Score=ranking["Score"].values, Rank=ranking["Rank"].values)
)
df_ranked[["name", "category", "price", "rating", "Score", "Rank"]].head(10)

,name,category,price,rating,Score,Rank
23,MANLI GALLARDO NVIDIA GeForce RTX 5080 OC 16GB...,gpu,30000000,5.0,0.5873,1
288,Lexar SSD NM610 PRO M.2 NVMe PCIe Gen 3x4 - 2TB,ssd,6123200,5.0,0.6161,2
311,Lexar NM620 M.2 Pcie Gen3 Nvme 2280 256GB - SS...,ssd,1180000,5.0,0.6347,3
103,PC Mini Deskmeet B660 - Intel core i5 13400 - ...,gpu,14153000,5.0,0.6412,4
9,VGA ASUS Dual GeForce RTX 5070 12GB GDDR7 OC E...,gpu,16999000,5.0,0.6671,5
12,MSI GeForce RTX 5070 VENTUS 2X OC 12GB GDDR7 |...,gpu,19033000,5.0,0.6658,6
18,VGA Card MSI GeForce RTX 5060 Ti 8G GAMING OC ...,gpu,12448000,5.0,0.7123,7
33,GAINWARD NVIDIA GEFORCE RTX 5080 16GB PHOENIX ...,gpu,29450000,5.0,0.5902,8
45,Zotac NVIDIA Geforce RTX 5060 Ti Twin Edge OC ...,gpu,10550000,5.0,0.7015,9
58,ZOTAC GAMING GeForce RTX 5070 SOLID OC 12GB GDDR7,gpu,16569000,5.0,0.6672,10


## 9. Visualization

In [10]:
from src.visualization import Visualizer

viz = Visualizer(
    df,
    output_dir="outputs/visualizations",
    **{k: v for k, v in config["visualization"].items() if k in ("style", "palette", "dpi")},
)
_ = viz.plot_price_trends()
_ = viz.plot_correlation_heatmap()
_ = viz.plot_ranking_bar_chart(ranking)
print("Charts saved to outputs/visualizations/")

Charts saved to outputs/visualizations/


## 10. Export Results

In [11]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)
df.to_csv(out / "cleaned_data.csv", index=False)
ranking.to_csv(out / "rankings.csv", index=False)
print("Saved:", [p.name for p in out.iterdir()])

Saved: ['cleaned_data.csv', 'prediction.json', 'rankings.csv', 'reviews_sample.csv', 'run_summary.json', 'visualizations']


## 11. Conclusions

- **Why prices rose:** fill in after running against live data.
- **Top value pick:** see ranking table above.
- **Normalization outlook:** see `docs/03-methodology.md` Phase 6 scenarios.

*Replace the demo sample/sentiment data with real scraped data and labelled reviews for production-grade results.*